# [3] LangChain Components

In [ ]:
!pip install -q langchain
!pip install -q langchain-openai

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path = "/content/.env", override=True)

print(".env 내 OPENAI_API_KEY가 환경변수에 할당됐습니다:", os.environ["OPENAI_API_KEY"][:5]+"*****")
print(".env 내 TAVILY_API_KEY가 환경변수에 할당됐습니다:", os.environ["TAVILY_API_KEY"][:5]+"*****")

#1. Chat Model



In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

## 1-1. 파라미터

temperature (0.0 ~ 2.0) : 답변의 무작위성과 창의성을 조절. 목적에 맞게 설정 필요

  - 값이 0에 가까울수록 가장 확률이 높은 단어만 일관되게 선택
  - 값이 높을수록 다양한 단어를 선택하여 창의적이고 예측 불가능한 답변

```
temperature를 0.0 ~ 0.1 : 팩트 위주로 정확하게 답하도록 통제
temperature를 0.7 ~ 1.0 : 모델의 상상력을 적극 활용
```

In [ ]:
# 1. Temperature가 0일 때 (보수적, 일관성 위주)
model_temp_0 = init_chat_model("gpt-5-nano", temperature=0.0)
print("[Temperature 0.0]")
print(model_temp_0.invoke("K-뷰티 홍보 문구를 간략하게 작성해줘요.").content)

# 2. Temperature가 1.0일 때 (창의적, 무작위성 위주)
model_temp_1 = init_chat_model("gpt-5-nano", temperature=1.0)
print("\n[Temperature 1.0]")
print(model_temp_1.invoke("K-뷰티 홍보 문구를 간략하게 작성해줘요.").content)

Timeout

(장애 방지) 모델 호출 후 응답을 얼마나 기다릴지(초 단위) 설정

서비스 운영 시 일시적 지연이 발생할 수 있으며, 이때 timeout이 없으면 서버의 스레드도 무한정 대기하다가 전체 서비스 장애로 이어질 수 있음

timeout=30처럼 명시적인 제한을 두어 빠른 실패를 유도하고 재시도 로직을 태우는 것이 안전 함



---


Max Tokens

(비용 제한) 모델이 생성할 수 있는 응답의 최대 길이(토큰 수)를 제한

악의적인 사용자의 프롬프트 인젝션이나 모델의 무한 반복 오류로 인해 불필요하게 긴 답변이 생성되는 것을 막아주는 최소한의 비용 방어막

단, 요약이나 번역처럼 본질적으로 긴 출력이 필요한 작업에서는 너무 낮게 잡지 않도록 주의 필요

In [ ]:
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model # init_chat_model 임포트

# 1. init_chat_model을 사용한 모델 통합 초기화
# - timeout: 10초 동안 모델 응답이 없으면 에러를 발생시킵니다 (네트워크 지연 방지).
# - max_tokens: 모델이 생성할 답변의 최대 토큰 수를 50개로 제한합니다 (비용 및 환각 방지).
model = init_chat_model(
    "gpt-4o-mini",                    # 모델 이름을 첫 번째 인자로 전달
    temperature=0.7,                  # 답변의 무작위성과 창의성을 조절
    timeout=10.0,                     # 초 단위 설정
    max_tokens=50                     # 생성 토큰 수 제한
)

response = model.invoke("인공지능의 정의를 한 문장으로 요약해줘.")

In [ ]:
response

## 1-2. 모델 호출

invoke()

가장 기본이 되는 호출 방식.

입력(질문/지시)을 넣으면 전체 생성이 끝난 후 응답 객체가 돌아 옴


In [ ]:
response = model.invoke("안녕하세요. 당신은 누구입니까?")
print(response.content)

stream()

모델이 답변을 생성하는 즉시 Chunk단위로 결과를 반환.

사용자가 첫 번째 단어를 보는 데 걸리는 시간을 줄일 수 있어 사용자 경험(UX)이 극적으로 개선


In [ ]:
# stream()은 chunk들의 제너레이터(generator)를 반환
for chunk in model.stream("AI Agent란 무엇인지 1000자 이상으로 설명해 주세요"):
    # chunk는 AIMessageChunk 객체이므로 .content로 텍스트를 추출
    print(chunk.content, end="", flush=True)

batch()

여러 개의 입력을 병렬로 동시 처리. (스레드 풀 등을 활용해 동시에 API 요청을 하기에, 10개의 질문도 5~10초 내외로 응답을 받을 수 있음)

대량의 문서를 요약하거나 수많은 데이터 행을 일괄 변환해야 하는 데이터 전처리 파이프라인에서 강력한 무기로 활용

(주의) 수십, 수백 개의 데이터를 한 번에 batch()로 넘기면, API 제공자(OpenAI 등) 측에서 초당 요청 제한 초과로 에러를 발생시킬 수 있으므로 config 파라미터를 통해 동시에 실행될 최대 작업 수를 제어 함


In [ ]:
inputs = [
    "과적합(Overfitting)이 뭔가요? 한 줄로 요약해줘.",
    "앵무새의 털 색상이 화려한 이유를 한 줄로 요약해줘.",
    "AI Agent의 핵심 특징 한 가지는?",
    "오로라의 현상을 한 줄로 요약해줘",
    "LangChain이 AI Agent 개발자에게 제공하는 주요 핵심 기능을 한 줄로 요약해줘.",
    "LangChain과 LangGraph의 차별성을 한 줄로 요약해줘"
]

# 한 번의 호출로 6개의 질문을 병렬 처리
responses = model.batch(inputs)

for i, response in enumerate(responses):
    print(f"[{i+1}번 답변] {response.content}")

In [ ]:
# 최대 3개씩만 동시에 처리하도록 제한 (Rate Limit 방어)
responses = model.batch(inputs, config={"max_concurrency": 3})

for i, response in enumerate(responses):
    print(f"[{i+1}번 답변] {response.content}")

#2. Message


## 2-1. 메시지 객체 활용

langchain.messages 객체를 활용하여 message 정의

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

system_msg = SystemMessage("당신은 유능한 로켓 전문가입니다.")
human_msg = HumanMessage("안녕하세요. 궁금한 게 있어요!")

messages = [system_msg, human_msg]
response = model.invoke(messages)
print(response.content)

In [ ]:
messages = [
    SystemMessage("당신은 친절한 조교입니다."),
    HumanMessage("안녕하세요. 저는 Jumany라고 합니다."),
    AIMessage("안녕하세요 Jumany님, 반갑습니다. 무엇을 도와드릴까요?"),
    HumanMessage("제가 방금 제 이름을 뭐라고 했죠?"),
]

response = model.invoke(messages)
print(response.content)

## 2-2. 딕셔너리 활용

Role, content를 정의

Role : “system”, “human” or “user”, “ai” or “assistant”

In [ ]:
messages = [
    {"role": "system", "content": "당신은 유능한 로켓 전문가입니다."},
    {"role": "human", "content": "안녕하세요. 궁금한 게 있어요!"},
    {"role": "ai", "content": "로켓 관련 무엇이든 물어보세요."},
    {"role": "human", "content": "추진 방식 차이를 설명해 주세요"},
]

response = model.invoke(messages)

In [ ]:
response.content

#3. Prompt Template

## 3-1. from_template() 메소드를 사용하여 PromptTemplate 객체 생성

In [ ]:
from langchain_core.prompts import PromptTemplate

# template 정의. {country}는 변수로, 이후에 값이 들어갈 자리를 의미
template = "{country}의 수도는 어디인가요?"

# from_template 메소드를 이용하여 PromptTemplate 객체 생성
prompt = PromptTemplate.from_template(template)
print(f"### prompt : \n{prompt}\n")

# prompt 생성. format 메소드를 이용하여 변수에 값을 넣어주면, 일회성으로 프롬프트 문자열이 생성
# 이 프롬프트 객체(prompt)를 재할당하지 않고, formatted_prompt와 같이 별도의 변수에 저장하거나 바로 출력하는 것이 좋음
formatted_prompt_string = prompt.format(country="대한민국")
print(f"### formatted_prompt_string : \n{formatted_prompt_string}\n")


In [ ]:
# chain 생성
chain = prompt | model

# country 변수에 입력된 값이 자동으로 치환되어 수행됨
# invoke 메서드는 딕셔너리 형태의 입력을 기대
chain.invoke({"country": "대한민국"}).content

## 3-2. PromptTemplate 객체 생성과 동시에 prompt 생성

`PromptTemplate`

- 사용자의 입력 변수를 사용하여 완전한 프롬프트 문자열을 만드는 데 사용되는 템플릿입니다
- 사용법
  - `template`: 템플릿 문자열입니다. 이 문자열 내에서 중괄호 `{}`는 변수를 나타냅니다.
  - `input_variables`: 중괄호 안에 들어갈 변수의 이름을 리스트로 정의합니다.

`input_variables`

- input_variables는 PromptTemplate에서 사용되는 변수의 이름을 정의하는 리스트입니다.

In [ ]:
# template 정의
template = "{country}의 수도는 어디인가요?"

# PromptTemplate 객체를 활용하여 prompt_template 생성
prompt = PromptTemplate(
    template=template,
    input_variables=["country"],
)

# prompt 생성
formatted_prompt_string = prompt.format(country="대한민국")

In [ ]:
print(f"### formatted_prompt_string : \n{formatted_prompt_string}\n")

In [ ]:
# chain 생성
chain = prompt | model

# country 변수에 입력된 값이 자동으로 치환되어 수행됨
# invoke 메서드는 딕셔너리 형태의 입력을 기대
chain.invoke({"country": "프랑스"}).content

## 3-3. load_prompt() : 파일로부터 template 읽어오기

1. test 폴더 만들기
2. yaml 파일 업로드

In [ ]:
from langchain_core.prompts import load_prompt

prompt = load_prompt("./test/fruit_color.yaml")

In [ ]:
# chain 생성
chain = prompt | model

chain.invoke({"fruit": "사과"}).content

In [ ]:
from langchain_core.prompts import load_prompt

prompt = load_prompt("./test/capital.yaml")

In [ ]:
# chain 생성
chain = prompt | model

chain.invoke({"country": "영국"}).content

## 3-4. ChatPromptTemplate : 대화목록을 프롬프트로 주입하고자 할 때 활용

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, AIMessagePromptTemplate

# MessagePromptTemplate 객체를 활용한 템플릿 정의
chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("당신은 친절한 AI 어시스턴트입니다. 당신의 이름은 {name} 입니다."),
        HumanMessagePromptTemplate.from_template("반가워요!"),
        AIMessagePromptTemplate.from_template("안녕하세요! 무엇을 도와드릴까요?"),
        HumanMessagePromptTemplate.from_template("{user_input}"),
    ]
)

# 챗 message 생성
messages = chat_template.format_messages(
    name="하니", user_input="당신의 이름은 무엇입니까?"
)

In [ ]:
messages

In [ ]:
# chain 생성
chain = chat_template | model

response = chain.invoke({"name": "하니", "user_input": "당신의 이름은 무엇입니까?"})
print(response.content)

참고. MessagesPlaceholder

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# MessagesPlaceholder는 이전 대화 목록(List of Messages)이 들어갈 '빈 자릿표' 역할을 합니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 가상 RPG 게임의 친절한 안내원 '모험가 가이드'입니다."),
    # ★ 이전 대화 기록이 들어갈 위치 지정
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

chain = prompt | model

history = [
    HumanMessage(content="안녕! 내 캐릭터 이름은 '용사키우기'야."),
    AIMessage(content="반갑습니다, '용사키우기'님! 무엇을 도와드릴까요?")
]

response = chain.invoke({
    "chat_history": history,  # MessagesPlaceholder 자리에 이 리스트가 삽입됨
    "input": "내 이름을 기억하고 있니?"
})


In [ ]:
response.content

#4. Structured Output

##4-1. Pydantic
- 파이썬 환경에서 에이전트를 개발한다면 압도적으로 추천하는 방식
- 클래스 형태로 구조를 잡기 때문에 코드가 깔끔하고, IDE의 자동완성 지원을 받을 수 있음

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, Literal

class Movie(BaseModel):
    """상세한 영화 정보."""
    title: str = Field(description="영화의 제목 (예: 인셉션)")
    year: Optional[int] = Field(default=None, description="개봉 연도. 정보를 알 수 없다면 None.")
    genre: Literal["액션", "로맨스", "SF", "코미디", "기타"] = Field(description="영화의 장르")
    director: str = Field(description="영화 감독 이름")
    rating: float = Field(description="영화 평점 (10점 만점 기준)")

In [ ]:
# 1. 스키마를 전달하여 구조화된 모델 생성
model_with_structure = model.with_structured_output(Movie)

# 2. 자연어 프롬프트로 호출
response = model_with_structure.invoke("영화 도둑들에 대해 설명해 주세요")

# 3. 결과 확인
print(response)

In [ ]:
print(f"제목: {response.title} (타입: {type(response.title)})")
print(f"평점: {response.rating} (타입: {type(response.rating)})")

##4-2. JSON Schema
- JSON Schema는 스키마를 딕셔너리로 정의하는 방식
- 파이썬에 종속되지 않기 때문에 언어 중립적인 스키마를 만들 때 유용

In [ ]:
import json

json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "director", "rating"]
}

In [ ]:
# 1. JSON 스키마를 전달
model_with_structure = model.with_structured_output(json_schema)

# 2. 호출
response = model_with_structure.invoke("영화 극한직업에 대해서 소개해 주세요")

# 3. 결과 확인
print(response)

In [ ]:
print(response['title'])
print(response['director'])

#5. LCEL(LangChain Expression Language)

기본 예시: 프롬프트 + 모델 + 출력 파서

가장 기본적이고 일반적인 사용 사례는 prompt 템플릿과 모델을 함께 연결하는 것

```
chain = prompt | model | output_parser
```

`|` 기호는 [unix 파이프 연산자](<https://en.wikipedia.org/wiki/Pipeline_(Unix)>)와 유사하며, 서로 다른 구성 요소를 연결하고 한 구성 요소의 출력을 다음 구성 요소의 입력으로 전달

이 체인에서 사용자 입력은 프롬프트 템플릿으로 전달되고, 그런 다음 프롬프트 템플릿 출력은 모델로 전달

## 5-1. Prompt | Model

In [ ]:
from langchain_core.prompts import PromptTemplate

# prompt 를 PromptTemplate 객체로 생성합니다.
prompt = PromptTemplate.from_template("{topic} 에 대해 쉽게 설명해주세요.")

model = init_chat_model(model="gpt-4.1-nano", temperature=0.1)

# 프롬프트, 모델, 출력 파서를 연결하여 처리 체인을 구성합니다.
chain = prompt | model

Chain 실행

- python 딕셔너리 형태로 입력값을 전달합니다.(키: 값)
- invoke() 함수 호출 시, 입력값을 전달합니다.

In [ ]:
# input 딕셔너리에 주제를 설정
input = {"topic": "인공지능 모델의 학습 원리"}

# chain 객체의 invoke 메서드를 사용하여 input을 전달합니다.
chain.invoke(input)

아래는 스트리밍을 출력하는 예시 입니다.

In [ ]:
# 스트리밍 출력을 위한 요청
answer = chain.stream(input)
# 스트리밍 출력
for chunk in answer:
    print(chunk.content, end="", flush=True)

## 5-2. Prompt | Model | OutputParser

출력파서(Output Parser)


In [ ]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

Chain 에 출력파서를 추가합니다.

In [ ]:
# 프롬프트, 모델, 출력 파서를 연결하여 처리 체인을 구성합니다.
chain = prompt | model | output_parser

In [ ]:
# chain 객체의 invoke 메서드를 사용하여 input을 전달합니다.
input = {"topic": "인공지능 모델의 학습 원리"}
chain.invoke(input)

In [ ]:
# 스트리밍 출력을 위한 요청
answer = chain.stream(input)
# 스트리밍 출력
for chunk in answer:
    print(chunk, end="", flush=True)

## 5-3. 영어회화 예시

In [ ]:
template = """
당신은 영어를 가르치는 10년차 영어 선생님입니다. 주어진 상황에 맞는 영어 회화를 작성해 주세요.
양식은 [FORMAT]을 참고하여 작성해 주세요.

#상황:
{question}

#FORMAT:
- 영어 회화:
- 한글 해석:
"""

# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다.
model = init_chat_model("gpt-4.1-nano")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

In [ ]:
# 체인을 구성합니다.
chain = prompt | model | output_parser

In [ ]:
# 완성된 Chain을 실행하여 답변을 얻습니다.
print(chain.invoke({"question": "저는 식당에 가서 음식을 주문하고 싶어요"}))

In [ ]:
answer = chain.stream({"question": "저는 식당에 가서 음식을 주문하고 싶어요"})
# 스트리밍 출력
for chunk in answer:
    print(chunk, end="", flush=True)

In [ ]:
answer = chain.stream({"question": "미국에서 피자 주문"})
# 스트리밍 출력
for chunk in answer:
    print(chunk, end="", flush=True)

## 5-4. 인터페이스

###① chain 생성

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

# 1. 주어진 토픽에 대한 농담을 요청하는 프롬프트 템플릿을 생성
prompt = PromptTemplate.from_template("{topic} 에 대하여 3문장으로 설명해줘.")
# 2. ChatModel을 인스턴스화
model = init_chat_model("gpt-4.1-nano")
# 3. 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()
# 프롬프트와 모델을 연결하여 대화 체인을 생성
chain = prompt | model | output_parser

###② stream : 실시간 출력

In [ ]:
# chain.stream 메서드를 사용하여 '멀티모달' 토픽에 대한 스트림을 생성하고 반복합니다.
for token in chain.stream({"topic": "멀티모달"}):
    # 스트림에서 받은 데이터의 내용을 출력합니다. 줄바꿈 없이 이어서 출력하고, 버퍼를 즉시 비웁니다.
    print(token, end="", flush=True)

###③ invoke : 출력

In [ ]:
# chain 객체의 invoke 메서드를 호출하고, 'ChatGPT'라는 주제로 딕셔너리를 전달합니다.
chain.invoke({"topic": "ChatGPT"})


###④ batch : 배치 (단위 실행)

In [ ]:
# 주어진 토픽 리스트를 batch 처리하는 함수 호출
# chain.batch([{"topic": "ChatGPT"}, {"topic": "Instagram"}])

chain.batch(
    [
        {"topic": "ChatGPT"},
        {"topic": "Instagram"},
        {"topic": "멀티모달"},
        {"topic": "프로그래밍"},
        {"topic": "머신러닝"},
    ],
    config={"max_concurrency": 3},
)

###⑤ async_stream : 비동기 stream

In [ ]:
# 비동기 스트림을 사용하여 'YouTube' 토픽의 메시지를 처리합니다.
async for token in chain.astream({"topic": "YouTube"}):
    # 메시지 내용을 출력합니다. 줄바꿈 없이 바로 출력하고 버퍼를 비웁니다.
    print(token, end="", flush=True)

###⑥ async_invoke : 비동기 호출

In [ ]:
# 비동기 체인 객체의 'ainvoke' 메서드를 호출하여 'NVIDIA' 토픽을 처리합니다.
my_process = chain.ainvoke({"topic": "NVIDIA"})

# 비동기로 처리되는 프로세스가 완료될 때까지 기다립니다.
await my_process

###⑦ async_batch : 비동기 배치

In [ ]:
# 주어진 토픽에 대해 비동기적으로 일괄 처리를 수행합니다.
my_abatch_process = chain.abatch(
    [{"topic": "YouTube"},
     {"topic": "Instagram"},
     {"topic": "Facebook"}]
)

# 비동기로 처리되는 일괄 처리 프로세스가 완료될 때까지 기다립니다.
await my_abatch_process

###⑧ parallel : 병렬성

In [ ]:
from langchain_core.runnables import RunnableParallel

# {country} 의 수도를 물어보는 체인을 생성합니다.
chain1 = (
    PromptTemplate.from_template("{country} 의 수도는 어디야?")
    | model
    | StrOutputParser()
)

# {country} 의 면적을 물어보는 체인을 생성합니다.
chain2 = (
    PromptTemplate.from_template("{country} 의 면적은 얼마야?")
    | model
    | StrOutputParser()
)

# 위의 2개 체인을 동시에 생성하는 병렬 실행 체인을 생성합니다.
combined = RunnableParallel(capital=chain1, area=chain2)


In [ ]:
# chain1 (수도) 를 실행합니다.
chain1.invoke({"country": "대한민국"})

In [ ]:
# chain2 (국토 면적) 를 실행합니다.
chain2.invoke({"country": "미국"})

In [ ]:
# 병렬 (수도, 국토 면적) 실행 체인을 실행합니다.
combined.invoke({"country": "대한민국"})

In [ ]:
# 배치 처리를 수행합니다.
chain1.batch([{"country": "대한민국"}, {"country": "미국"}])

In [ ]:
# 배치 처리를 수행합니다.
chain2.batch([{"country": "대한민국"}, {"country": "미국"}])

In [ ]:
# 주어진 데이터를 배치로 처리합니다.
combined.batch([{"country": "대한민국"}, {"country": "미국"}])

## 5-5. Runnable

###① 예시 1

RunnableParallel, RunnablePassthrough

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

runnable = RunnableParallel(
    # 전달된 입력을 그대로 반환하는 Runnable을 설정합니다.
    passed=RunnablePassthrough(),
    # 입력의 "num" 값에 3을 곱한 결과를 반환하는 Runnable을 설정합니다.
    extra=RunnablePassthrough.assign(mult=lambda x: x["num"] * 3),
    # 입력의 "num" 값에 1을 더한 결과를 반환하는 Runnable을 설정합니다.
    modified=lambda x: x["num"] + 1,
)

# {"num": 1}을 입력으로 Runnable을 실행합니다.
runnable.invoke({"num": 1})


###② 예시 2

RunnableParallel, RunnablePassthrough

In [ ]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

prompt1  = ChatPromptTemplate.from_messages([("system", "긍정적인 측면으로 답하세요."), ("human", "{q}")])
prompt2  = ChatPromptTemplate.from_messages([("system", "부정적인 측면으로 답하세요."), ("human", "{q}")])

model = init_chat_model("gpt-4.1-nano")

parser = StrOutputParser()

chain1 = prompt1 | model | parser
chain2 = prompt2 | model | parser

act = RunnableParallel(
    question=RunnablePassthrough() | itemgetter("q"),  # 입력을 통과시킨 뒤 q 값만 추출
    positive=chain1,
    negative=chain2,
)

out = act.invoke({"q": "2배 레버리지 ETF 투자에 대해 어떻게 생각하세요?"})


In [ ]:
print("Question:", out["question"])
print("*** Answer POSITIVE:", out["positive"][:70], "...")
print("*** Answer NEGATIVE:", out["negative"][:70], "...")

In [ ]:
out

###③ 예시 3

RunnableLambda

In [ ]:
def combine_text(text):
    return f"positive : {text["positive"]}  negative : {text["negative"]}"

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

sum_prompt = ChatPromptTemplate.from_messages([("system", "다음 내용을 자연스럽게 200자 이내로 교정해줘요"), ("human", "{info}")])

sum_act = act | {"info": RunnableLambda(combine_text)} | sum_prompt | model | StrOutputParser()


In [ ]:
sum_out = sum_act.invoke({"q": "2배 레버리지 ETF 투자에 대해 어떻게 생각하세요?"})


In [ ]:
sum_out

###④ 예시 4

RunnableBranch, RunnablePassthrough

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """주어진 사용자 질문을 `수학`, `과학`, 또는 `기타` 중 하나로 분류하세요. 한 단어 이상으로 응답하지 마세요.
       질문: {question}
       Classification:"""
)

# 체인을 생성합니다.
chain = (
    prompt
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()  # 문자열 출력 파서를 사용합니다.
)


In [ ]:
# 질문을 입력하여 체인을 호출합니다.
chain.invoke("2+2 는 무엇인가요?")

# chain.invoke("작용 반작용의 법칙은 무엇인가요?")

# chain.invoke("Google은 어떤 회사인가요?")


In [ ]:
math_chain = (
    PromptTemplate.from_template(
        """You are an expert in math. \
Always answer questions starting with "파스칼선생님께서 말씀하시기를..". \
Respond to the following question:

Question: {question}
Answer:"""
    )
    # OpenAI의 LLM을 사용합니다.
    | ChatOpenAI(model="gpt-4o-mini")
)

science_chain = (
    PromptTemplate.from_template(
        """You are an expert in science. \
Always answer questions starting with "뉴턴선생님께서 말씀하시기를..". \
Respond to the following question:

Question: {question}
Answer:"""
    )
    # OpenAI의 LLM을 사용합니다.
    | ChatOpenAI(model="gpt-4o-mini")
)

general_chain = (
    PromptTemplate.from_template(
        """Respond to the following question concisely:

Question: {question}
Answer:"""
    )
    # OpenAI의 LLM을 사용합니다.
    | ChatOpenAI(model="gpt-4o-mini")
)


In [ ]:
# RunnableBranch 로 분기
from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    # 주제에 "수학"이 포함되어 있는 경우, math_chain을 실행합니다.
    (lambda x: "수학" in x["topic"].lower(), math_chain),
    # 주제에 "과학"이 포함되어 있는 경우, science_chain을 실행합니다.
    (lambda x: "과학" in x["topic"].lower(), science_chain),
    # 그 외의 경우 general_chain을 실행합니다.
    general_chain,
)

from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

full_chain = (
    {"topic": chain, "question": RunnablePassthrough()}
    | branch
    | StrOutputParser()
)

In [ ]:
# 수학과 관련된 질문을 입력하여 체인을 호출합니다.
full_chain.invoke("미적분의 개념에 대해 말씀해 주세요.")

In [ ]:
# 과학과 관련된 질문을 입력하여 체인을 호출합니다.
full_chain.invoke("중력은 어떻게 작용하나요?")

In [ ]:
# 기타 질문을 입력하여 체인을 호출합니다.
full_chain.invoke("RAG(Retrieval Augmented Generation)은 무엇인가요?")

#6. Tool Calling

## 6-1. Built-in tools

검색 도구 - Tavily
1. API 키 발급 -> https://app.tavily.com/
2. 발급한 API 키를 환경변수에 설정 (.env 파일)
      - .env파일 -> TAVILY_API_KEY=tvly-abcdefghijklmnopqrstuvwxyz

In [ ]:
!pip install -qU langchain-tavily

from langchain_tavily import TavilySearch

# 도구 생성
tool = TavilySearch(
    max_results=6,
    include_answer=True,
    include_raw_content=True,
    include_domains=["github.io", "wikidocs.net"],
)

In [ ]:
# 도구 실행
tool.invoke({"query": "LangChain Tools 에 대해서 알려주세요"})

## 6-2. Custom tools

In [ ]:
from langchain.tools import tool

# 데코레이터를 사용하여 함수를 도구로 변환합니다.
@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

In [ ]:
# 도구 실행
add_numbers.invoke({"a": 3, "b": 4})

In [ ]:
# 도구 실행
multiply_numbers.invoke({"a": 3, "b": 4})